In [ ]:
# Needed to import from the enderscope library

import os

os.chdir("..")


In [ ]:

from datetime import datetime as dt
from pathlib import Path

from tqdm import tqdm

import pandas as pd

from enderleaf.tools import ensure_folder, format_datetime, write_dataframe, TIME_FORMAT
from enderleaf.preview_panel import expand_file_path

In [ ]:
images = list(Path(".").joinpath("output", "images", "Exp26DM14", "I2").glob("*.png"))
print(len(images))
images[0].name

In [ ]:
df = pd.concat(
    [pd.DataFrame(expand_file_path(file)) for file in tqdm(images)]
).sort_values("date_time")
df

In [ ]:
for row in df[["exp", "inoc", "plate", "month", "day"]].drop_duplicates().itertuples():
    df_pd = (
        df[(df.plate == row.plate) & (df.day == row.day)]
        .sort_values("date_time")
        .reset_index(drop=True)
    )
    df_pd["job_ts"] = df_pd["date_time"].min()
    write_dataframe(
        df_pd,
        Path(".")
        .joinpath("output", "job_data", row.exp, "I" + row.inoc)
        .joinpath(
            row.exp
            + "#I"
            + str(row.inoc)
            + "#P"
            + str(row.plate)
            + "#"
            + df_pd.iloc[0].job_ts.strftime(TIME_FORMAT)
        )
        .with_suffix(".csv"),
    )

df_pd